# 06 · AWGN and performance metrics

Additive white Gaussian noise (AWGN) is the reference channel: thermal
noise, amplifier noise (ASE) and many small impairments together look
Gaussian.  It is also where every metric has a closed form to check against.
This notebook covers:

1. Es/N0, and how `apply_awgn` sets the noise at any oversampling;
2. SNR and EVM, and how they relate;
3. symbol and bit error rates against theory;
4. **soft information**: log-likelihood ratios (LLRs);
5. achievable rates: **GMI** and **MI** against the Shannon capacity;
6. conventions of the metric functions (skipped symbols, MIMO).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import erfc

import commkit as ck
from commkit import RRC, Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()


def host(a):
    return to_device(a, "cpu")


def q_function(x):
    return 0.5 * erfc(x / np.sqrt(2))

## 1. Es/N0 at any oversampling

`Es/N0` is the symbol energy over the noise spectral density.  For a
unit-energy symbol, the matched filter output at the symbol instant has
exactly `SNR = Es/N0`.  At `sps` samples per symbol the simulated band is
`sps` times the symbol rate, so `apply_awgn` spreads the noise power
accordingly: the waveform's sample SNR is `Es/N0 - 10 log10(sps)`, and the
matched filter brings it back.  `Eb/N0 = Es/N0 - 10 log10(k)` normalizes
per bit.

In [ ]:
tx = ck.generate(
    Constellation.qam(16),
    num_symbols=2**15,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
    device=DEVICE,
)
for esn0 in (10, 15, 20):
    rx = ck.impairments.apply_awgn(tx, esn0_db=esn0, rng=2)
    noise = host(rx.samples - tx.samples)
    sample_snr = 10 * np.log10(
        np.mean(np.abs(host(tx.samples)) ** 2) / np.mean(np.abs(noise) ** 2)
    )
    y = ck.multirate.decimate_to_symbol_rate(ck.filtering.matched_filter(rx))
    print(
        f"Es/N0 = {esn0} dB: sample SNR {sample_snr:5.2f} dB, after the matched filter {ck.metrics.snr(y):5.2f} dB"
    )

## 2. SNR and EVM

Both compare the received symbols `r` with the transmitted ones `s`:

* `snr` is `E|s|² / E|r/g - s|²` in dB, with the real gain
  `g = |⟨r s*⟩| / ⟨|s|²⟩` divided out, so a scaled output is not penalized
  for its scale (a rotation is: metrics never rotate);
* `evm` is the RMS error vector relative to the RMS symbol, in percent.
  For data-aided measurements `EVM = 1 / sqrt(SNR)`.  With `blind=True` the
  error is taken against the *nearest* point instead of the transmitted
  one, as an instrument without a reference does; at low SNR wrong decisions
  make the blind EVM read too low.

Simulations from here on mostly use one sample per symbol, where Es/N0 is
directly the per-symbol SNR.

In [ ]:
C16 = Constellation.qam(16)
sym = ck.generate(C16, num_symbols=2**16, symbol_rate=1e9, rng=3, device=DEVICE)
print(" Es/N0    SNR    EVM   1/sqrt(SNR)   blind EVM")
for esn0 in (6, 10, 14, 18, 22):
    r = ck.impairments.apply_awgn(sym, esn0_db=esn0, rng=4)
    s = ck.metrics.snr(r)
    print(
        f"{esn0:4d} dB {s:6.2f} {ck.metrics.evm(r):6.2f} %  {100 / np.sqrt(10 ** (s / 10)):6.2f} %"
        f"      {ck.metrics.evm(r, blind=True):6.2f} %"
    )

## 3. Error rates against theory

For square M-QAM with Gray labels, each axis is a `sqrt(M)`-PAM, and the
symbol error probability is

$$P_s = 1 - (1 - P_{\sqrt{M}})^2, \qquad
P_{\sqrt{M}} = 2\Big(1 - \tfrac{1}{\sqrt{M}}\Big)\, Q\!\left(\sqrt{\tfrac{3}{M-1}\tfrac{E_s}{N_0}}\right),$$

and since most errors hit a nearest neighbour, `BER ≈ P_s / log2 M`.
(For QPSK both are exact: `BER = Q(sqrt(2 Eb/N0))`.)  `ser` and `ber`
decide on the nearest point and count against the reference.  At low SNR
errors reach beyond the nearest neighbours and the `P_s / log2 M` curve
underestimates the BER; below about `1e-2` simulation and theory agree.

In [ ]:
esn0_grid = np.arange(0, 25, 2.0)
fig, ax = plt.subplots(figsize=(8, 5))
for M, color in [(4, "C0"), (16, "C1"), (64, "C2")]:
    c = Constellation.qam(M)
    sym = ck.generate(c, num_symbols=2**17, symbol_rate=1e9, rng=5, device=DEVICE)
    ber = [
        ck.metrics.ber(ck.impairments.apply_awgn(sym, esn0_db=e, rng=6))
        for e in esn0_grid
    ]
    esn0 = 10 ** (esn0_grid / 10)
    p_axis = 2 * (1 - 1 / np.sqrt(M)) * q_function(np.sqrt(3 / (M - 1) * esn0))
    ber_theory = (1 - (1 - p_axis) ** 2) / np.log2(M)
    ax.semilogy(esn0_grid, ber_theory, color=color, label=f"{M}-QAM theory")
    ax.semilogy(
        esn0_grid,
        np.where(np.array(ber) > 0, ber, np.nan),
        "o",
        color=color,
        label=f"{M}-QAM simulated",
    )
ax.set_ylim(1e-6, 0.5)
ax.set_xlabel("Es/N0 [dB]")
ax.set_ylabel("BER")
ax.legend()
plt.show()

## 4. Soft information: LLRs

A hard decision throws away how *sure* it was.  A soft-decision decoder
wants, for each bit `b`, the log-likelihood ratio

$$L_b = \log \frac{P(b = 0 \mid r)}{P(b = 1 \mid r)},$$

positive for a likely 0, large in magnitude for a reliable bit.  For
Gaussian noise of variance `σ²` it is a sum over the points labelled 0 and
1; `method="maxlog"` keeps only the nearest point of each set (the usual
receiver approximation), `"exact"` sums them all.  `compute_llr`
(or `Constellation.llr`) returns the `k` LLRs of each symbol next to each
other, and needs `noise_var` on the constellation's scale.

The LLRs of the first bit of 16-QAM at 12 dB: their sign is the hard
decision.  Each sign has two modes, because that bit is decided by the
in-phase axis: points of the inner columns lie next to the decision
boundary and get small LLRs, points of the outer columns large ones.

In [ ]:
esn0 = 12.0
sym = ck.generate(C16, num_symbols=2**15, symbol_rate=1e9, rng=7, device=DEVICE)
r = ck.impairments.apply_awgn(sym, esn0_db=esn0, rng=8)
noise_var = 10 ** (-esn0 / 10)
llr = host(ck.mapping.compute_llr(r, noise_var=noise_var))
bits = host(sym.reference.bits)

hard = (llr < 0).astype(bits.dtype)
print(
    f"sign of the LLR = hard decision: {np.array_equal(hard, host(ck.mapping.demap_symbols(r)))}"
)
print(f"BER from the LLR signs: {np.mean(hard != bits):.2e}")

first = slice(0, None, 4)  # the most significant bit of every symbol
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(
    llr[first][bits[first] == 0],
    bins=100,
    alpha=0.6,
    density=True,
    label="transmitted 0",
)
ax.hist(
    llr[first][bits[first] == 1],
    bins=100,
    alpha=0.6,
    density=True,
    label="transmitted 1",
)
ax.set_xlabel("LLR of bit 0")
ax.set_ylabel("density")
ax.set_title(f"16-QAM, Es/N0 = {esn0:.0f} dB, max-log LLRs")
ax.legend()
plt.show()

## 5. Achievable rates: GMI and MI

How many information bits per symbol can a code carry over this channel?

* **MI** (mutual information) `I(X; Y)` is the rate of an ideal decoder
  that works on whole symbols.
* **GMI** (generalized mutual information) is the rate of a binary decoder
  fed with the LLRs (bit-metric decoding), what practical FEC achieves.
  `GMI ≤ MI`, with a small gap for Gray-labelled QAM.
* Both stay below the AWGN capacity `log2(1 + SNR)`.

A **shaped** constellation (01 · section 5) closes part of the gap to
capacity at intermediate SNR.

In [ ]:
esn0_grid = np.arange(0, 26, 2.0)
formats = {
    "16-QAM": C16,
    "64-QAM": Constellation.qam(64),
    "PS-64QAM, H = 5": Constellation.qam(64).shaped(entropy=5.0),
}
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    esn0_grid,
    np.log2(1 + 10 ** (esn0_grid / 10)),
    "--",
    color="gray",
    label="capacity log2(1 + SNR)",
)
for i, (name, c) in enumerate(formats.items()):
    sym = ck.generate(c, num_symbols=2**14, symbol_rate=1e9, rng=9, device=DEVICE)
    g, m = [], []
    for e in esn0_grid:
        r = ck.impairments.apply_awgn(sym, esn0_db=e, rng=10)
        g.append(ck.metrics.gmi(r, noise_var=10 ** (-e / 10)))
        m.append(ck.metrics.mi(r, noise_var=10 ** (-e / 10)))
    ax.plot(esn0_grid, g, "o-", color=f"C{i}", label=f"{name} GMI")
    ax.plot(esn0_grid, m, ":", color=f"C{i}", label=f"{name} MI")
ax.set_xlabel("Es/N0 [dB]")
ax.set_ylabel("bits per symbol")
ax.legend()
plt.show()

## 6. Conventions of the metrics

* Metrics return **host values**: a `float` for `(N,)` input, a `(C,)`
  array for `(C, N)` input.
* `num_skip_symbols` leaves out the start of the record (equalizer training,
  convergence); an empty selection raises instead of returning a misleading
  number.
* Metrics never align, rotate or permute on their own: an output with a
  delay, a 90° rotation or swapped polarizations scores badly until the
  receiver resolves it (09, 11 and 16 show the tools).

In [ ]:
dual = ck.generate(
    C16, num_symbols=2**14, symbol_rate=1e9, num_channels=2, rng=11, device=DEVICE
)
r = ck.impairments.apply_awgn(dual, esn0_db=15, rng=12)
print("SNR per channel:", ck.metrics.snr(r), "dB")
print("EVM per channel:", ck.metrics.evm(r, num_skip_symbols=1000), "%")

rotated = r.replace(samples=1j * r.samples)
print("BER with a 90 degree rotation:", ck.metrics.ber(rotated))

**Summary**

* `apply_awgn(esn0_db=...)` gives SNR = Es/N0 after the matched filter at
  any `sps`.
* `snr` and data-aided `evm` are two views of one number,
  `EVM = 1/sqrt(SNR)`.
* `ser` / `ber` match the closed-form QAM curves; LLR signs are the hard
  decisions and their magnitudes the reliabilities.
* `gmi` is the rate a binary FEC can reach; it needs `noise_var`.

**Next:** [07 · Channel coding](07-channel-coding.ipynb).